In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz

dbutils.widgets.removeAll()
# Parámetros del Job
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

df_bronze = spark.table(f"{catalog}.bronze.tarjetas_bronze").filter(col("process_date") == process_date)

df_silver = df_bronze.select(
    col("id_tarjeta").cast("bigint"),
    col("id_cliente").cast("bigint"),
    col("tipo_tarjeta"),
    col("marca_tarjeta"),
    regexp_replace(col("limite_credito"), r"[\$,]", "").cast("double").alias("limite_credito"),
    months_between(current_date(), to_date(col("fecha_apertura"), "MM/yyyy")).cast("int").alias("antiguedad_meses"),
    when(col("en_dark_web") == "SÍ", True).otherwise(False).alias("tarjeta_en_riesgo"),
    current_timestamp().alias("ingestion_timestamp"),
    lit(process_date).alias("process_date")
).dropDuplicates(["id_tarjeta"])

target = f"{catalog}.silver.tarjetas_silver"
dt = DeltaTable.forName(spark, target)
dt.alias("t").merge(df_silver.alias("s"), "t.id_tarjeta = s.id_tarjeta") \
  .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_final_silver = spark.table(target)

total_registros_silver = df_final_silver.count()

registros_hoy = df_final_silver.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" REPORTE DE TABLA FÍSICA: {target.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Catálogo.Esquema.Tabla':<30} : {target}")
print(f"{'Total registros acumulados':<30} : {total_registros_silver:,}")
print(f"{'Registros cargados/actualizados hoy':<30} : {registros_hoy:,}")
print(f"{'Última sincronización (Perú)':<30} : {ts_peru}")
print("═" * 70)

print(f"\n🔍 VISTA PREVIA DE LA TABLA SILVER (Top 3 recientes):")

display(
    df_final_silver
    .orderBy(col("ingestion_timestamp").desc())
    .limit(3)
)